[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session1/notebooks/S1_NB02_voir_ecrire_sons.ipynb)

# S1-NB02 — Voir et écrire les sons : tons, durées, voyelles, Unicode

**Module 1 — Session 1 : Panorama de l'IA vocale & phonétique des langues locales** · Atelier 1B · 40 min · Ticket `M1-S1-05`

## 🎯 Objectifs
- **Voir** un ton : extraire et lire un contour de fréquence fondamentale (F0).
- Comparer des **paires minimales** (tons en fongbe ; longueur et gémination en wolof) par la mesure.
- Placer des voyelles réelles dans un **espace vocalique** (formants F1/F2).
- Comprendre pourquoi deux textes identiques à l'écran peuvent être **différents pour la machine** (Unicode, homoglyphes)
  et normaliser une transcription.

## 🧭 Comment travailler
| Badge | Pour qui | Attendu |
|---|---|---|
| 🟢 Essentiel | Tout le monde | Exécuter, observer, interpréter (sans coder) — ≈ 25 min |
| 🔵 Intermédiaire | À l'aise en Python | Compléter le code `✍️ À COMPLÉTER` |
| 🔴 Avancé | Experts ML | Défis ouverts |

💡 Profils linguistes : vous êtes les **experts** de cet atelier — guidez votre binôme dans l'interprétation.
En parallèle du parcours 🟢, vous pouvez faire la **partie B du TD1** (exercices d'API sur papier).

## 0. Préchauffage

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from m1utils.audio import load_audio, play
from m1utils.data import load_cached_predictions, load_manifest
from m1utils.features import f0_contour, f0_semitones, formants
from m1utils.preprocess import vad_segments
from m1utils.text import char_inventory, normalize_text, strip_tones
from m1utils.viz import compare_f0, plot_f0, plot_vowel_space, plot_waveform

paires = load_manifest("minimal_pairs")   # paires minimales (voix SYNTHÉTIQUE, en attendant vos enregistrements)
demo = load_manifest("demo_s1")           # énoncés réels (ALFFA, Kallaama)
display(paires[["id", "lang", "text", "contrast", "api", "gloss_fr"]])

> ⚠️ **À propos des paires minimales.** Ce pack est produit par **synthèse vocale** (MMS-TTS pour le fongbe,
> SpeechT5 pour le wolof), en attendant des enregistrements de locuteurs natifs — que **vous** allez réaliser au TP1.
> Nous allons donc mesurer, mais aussi **juger la synthèse** : réalise-t-elle correctement les tons et les durées ?
> Les exemples lexicaux sont à faire valider par des locuteurs [RELECTURE LINGUISTIQUE].

## 1. Voir la mélodie de la parole 🟢
La **F0** (fréquence fondamentale) est la vitesse de vibration des cordes vocales : c'est ce que l'oreille perçoit comme
la **hauteur** de la voix. Dans une langue à tons comme le fongbe, elle distingue des mots.

Ci-dessous : un énoncé fongbe **réel** (ALFFA). En gris le spectrogramme, en orange la F0 (absente pendant les silences
et les consonnes sourdes : pas de vibration des cordes vocales).

In [ ]:
enonce = demo[demo.lang == "fon"].iloc[0]
y, sr = load_audio(enonce.abspath)
play(y, sr, label=f"« {enonce.text} »")
plot_f0(y, sr, title=f"F0 de « {enonce.text} »")
plt.show()

**❓ Questions :**
1. Entre quelles valeurs (Hz) varie la F0 de ce locuteur ? Est-ce plutôt une voix grave ou aiguë ?
2. Pourquoi la courbe orange est-elle **interrompue** par endroits ?

✍️ *Votre réponse :* …

## 2. Paires minimales tonales (fongbe) 🟢
Deux mots qui ne diffèrent **que par le ton**. Écoutez, puis comparez les contours.

In [ ]:
fon = paires[paires.lang == "fon"]
for paire, groupe in fon.groupby("pair"):
    items = []
    for _, ligne in groupe.iterrows():
        signal, _ = load_audio(ligne.abspath)
        play(signal, label=f"{paire} : {ligne.text} ({ligne.gloss_fr})")
        items.append((f"{ligne.text}", signal))
    compare_f0(items)
    plt.title(f"Paire {paire} — {groupe.contrast.iloc[0]}")
    plt.show()

**❓ Questions :**
1. Pour quelles paires la mesure confirme-t-elle l'écriture (ton haut ◌́ plus aigu que ton bas ◌̀) ?
2. Pour lesquelles la synthèse semble-t-elle **ne pas** respecter le ton écrit ? Qu'en concluez-vous sur ce modèle de TTS ?

✍️ *Votre réponse :* …

### Exercice 2.1 🔵 — Résumer un contour par deux nombres
Pour chaque mot fongbe, calculez la **F0 moyenne** (Hz) et la **pente** (Hz/s) de la F0 sur les trames voisées
(indice : `np.polyfit(t, f0, 1)[0]` donne la pente d'une droite ajustée). Ignorez les mots sans F0 mesurable.

In [ ]:
def resume_f0(chemin):
    signal, sr = load_audio(chemin)
    t, f0 = f0_contour(signal, sr)
    voise = ~np.isnan(f0)
    if voise.sum() < 3:
        return np.nan, np.nan
    moyenne = ...  # ✍️ À COMPLÉTER
    pente = ...    # ✍️ À COMPLÉTER
    return moyenne, pente


fon_mesures = fon.assign(**dict(zip(["F0_moy_Hz", "pente_Hz_s"], zip(*fon.abspath.map(resume_f0)))))
display(fon_mesures[["pair", "text", "api", "F0_moy_Hz", "pente_Hz_s"]].round(1))

### Exercice 2.2 🔵 — Pourquoi parler en demi-tons ?
Un ton « haut » chez un homme peut être plus grave qu'un ton « bas » chez une femme. On exprime donc la F0 **relativement
au locuteur**, en demi-tons (12 demi-tons = une octave) : `f0_semitones(f0, ref_hz)`.
Convertissez 110 Hz et 220 Hz en demi-tons par rapport à 110 Hz, puis 200 Hz et 400 Hz par rapport à 200 Hz. Conclusion ?

In [ ]:
homme = ...  # ✍️ À COMPLÉTER : f0_semitones(np.array([110, 220]), ref_hz=110)
femme = ...  # ✍️ À COMPLÉTER
print("homme :", homme, "| femme :", femme)
assert np.allclose(homme, femme), "Les deux écarts doivent valoir 0 et 12 demi-tons."

✍️ *Votre réponse :* …

## 3. Longueur vocalique et gémination (wolof) 🟢
Le wolof n'a pas de tons, mais la **durée** distingue des mots : voyelle brève/longue (*fen* « mentir » / *feen*
« apparaître »), consonne simple/géminée (*gëm* « croire » / *gëmm* « fermer les yeux »).

In [ ]:
wol = paires[paires.lang == "wol"]
fig, axes = plt.subplots(len(wol) // 2, 2, figsize=(12, 2.2 * len(wol) // 2), sharex=True)
for ax, (_, ligne) in zip(axes.ravel(), wol.iterrows()):
    signal, sr = load_audio(ligne.abspath)
    segments = vad_segments(signal, sr, method="energy", min_speech_s=0.05, min_silence_s=0.1)
    plot_waveform(signal, sr, ax=ax, segments=segments, title=f"{ligne.text} ({ligne.gloss_fr})")
plt.tight_layout()
plt.show()

### Exercice 3.1 🔵 — Mesurer les durées
La zone orange est la parole détectée automatiquement (VAD). Calculez pour chaque mot la **durée de parole**
(somme des segments), puis le **rapport long/bref** pour chaque paire.

In [ ]:
def duree_parole(chemin):
    signal, sr = load_audio(chemin)
    segments = vad_segments(signal, sr, method="energy", min_speech_s=0.05, min_silence_s=0.1)
    return ...  # ✍️ À COMPLÉTER : somme des durées (fin - début) des segments


durees = wol.assign(duree_s=wol.abspath.map(duree_parole))
rapport = durees.pivot_table(index="pair", columns=durees.groupby("pair").cumcount(), values="duree_s")
rapport.columns = ["bref (s)", "long (s)"]
rapport["rapport long/bref"] = rapport["long (s)"] / rapport["bref (s)"]
display(rapport.round(2))

**❓ Question :** les mots longs/géminés sont-ils systématiquement plus longs ? Chez un locuteur humain, le rapport est
souvent proche de 1,5 à 2 pour une voyelle longue. Que pensez-vous de la synthèse ? Et de la mesure par VAD ?

✍️ *Votre réponse :* …

## 4. L'espace vocalique de vraies voix 🔵
Les **formants** F1 et F2 sont les résonances du conduit vocal : F1 dépend de l'ouverture de la bouche, F2 de la
position de la langue (avant/arrière). Chaque voyelle occupe une région du plan F1/F2.

Pour trouver les voyelles dans des **phrases réelles**, on utilise un modèle de reconnaissance (Simba-H) qui indique
**quand** chaque lettre est prononcée (alignement CTC, détaillé en session 3). Exécutez simplement les cellules.

In [ ]:
from m1utils.asr import ctc_char_alignment, load_asr, unload

modele = load_asr("simba-h")
VOYELLES = set("aeɛiouɔ")
mesures = []
for _, ligne in demo[demo.lang == "fon"].iterrows():
    signal, sr = load_audio(ligne.abspath)
    alignement = ctc_char_alignment(modele.logits(signal), modele.vocab, modele.blank_id)
    for c in alignement:
        lettre = normalize_text(c["char"], "fon", keep_tones=False)
        if lettre in VOYELLES:
            milieu = (c["start_s"] + c["end_s"]) / 2
            f = formants(signal, sr, at_s=milieu)
            mesures.append({"label": lettre, **f})
unload(modele)
voyelles = pd.DataFrame(mesures).dropna()
moyennes = voyelles.groupby("label")[["F1", "F2"]].median().reset_index()
display(moyennes.round(0))
plot_vowel_space(moyennes)
plt.show()

**❓ Question :** retrouvez-vous le « triangle » attendu : /i/ en haut à gauche (fermée, antérieure), /u/ en haut à droite
(fermée, postérieure), /a/ en bas (ouverte) ? Où se placent /ɛ/ et /ɔ/ par rapport à /e/ et /o/ ?

✍️ *Votre réponse :* …

## 5. Écrire les sons : Unicode et homoglyphes 🟢
Voici deux façons d'écrire « akwɛ » (argent, en fongbe). Elles sont **identiques à l'écran**.

In [ ]:
a = "akwɛ"   # ɛ LATIN (U+025B) — orthographe correcte
b = "akwε"   # ε GREC (U+03B5) — tel qu'écrit dans le corpus ALFFA !
print(a, b, "→ égales ?", a == b)
for mot in (a, b):
    print(mot, [f"U+{ord(c):04X} {unicodedata.name(c)}" for c in mot])

Et deux façons d'écrire « é » : précomposé (NFC) ou décomposé (NFD).

In [ ]:
nfc, nfd = unicodedata.normalize("NFC", "ɖé"), unicodedata.normalize("NFD", "ɖé")
print(nfc, nfd, "→ égales ?", nfc == nfd, "| longueurs :", len(nfc), len(nfd))
print("Après normalisation :", normalize_text(nfc, "fon") == normalize_text(nfd, "fon"))

**❓ Question :** un modèle de reconnaissance écrit « akwɛ » et la référence contient « akwε ». Combien d'erreurs un
calcul naïf compte-t-il ? Est-ce juste ?

✍️ *Votre réponse :* …

### Exercice 5.1 🔵 — Inventaire des caractères d'un corpus réel
Calculez l'inventaire des caractères des transcriptions fongbe du jeu de test de la session 3 avec
`char_inventory(textes, lang="fon")`, puis affichez les caractères **hors alphabet** (`in_alphabet == False`).

In [ ]:
test = load_manifest("testset_s3")
textes_fon = test[test.lang == "fon"].text.tolist()
inventaire = ...  # ✍️ À COMPLÉTER
display(inventaire[inventaire.in_alphabet == False])

### Exercice 5.2 🔵 — Les tons et la taille du vocabulaire
MMS écrit les tons. Comptez le nombre de **mots distincts** dans ses sorties fongbe sur le jeu de test (prédictions
pré-calculées), **avec** et **sans** tons. Pourquoi est-ce important pour un modèle de langue ou un lexique ?

In [ ]:
sorties_mms = load_cached_predictions("mms-1b-all", "fon").hyp.fillna("").tolist()
avec = ...  # ✍️ À COMPLÉTER : ensemble des mots, normalize_text(..., keep_tones=True)
sans = ...  # ✍️ À COMPLÉTER : idem avec keep_tones=False
print(f"Mots distincts avec tons : {len(avec)} | sans tons : {len(sans)}")
print("Exemples de mots qui fusionnent :",
      sorted({strip_tones(m, 'fon') for m in avec if strip_tones(m, 'fon') != m})[:10])

✍️ *Votre réponse :* …

## 6. TD1 partie B (intégrée) 🟢 — Transcrire en API
Écoutez les mots des paires minimales et proposez une transcription en Alphabet phonétique international (API).
Comparez avec la colonne `api` (cachée ci-dessous dans la version élève). Rappel : /ɡ͡b/ et /k͡p/ sont des
consonnes **labio-vélaires** (lèvres et voile du palais fermés en même temps), typiques des langues gbe.

In [ ]:
for _, ligne in paires.sample(5, random_state=1).iterrows():
    play(ligne.abspath, label=f"{ligne.lang} — « {ligne.text} » — votre transcription API : ……")

## 7. Défi 🔴 — Un détecteur de ton automatique
Écrivez `predire_ton(chemin)` qui renvoie `"haut"` ou `"bas"` à partir de la F0 en demi-tons relative à une référence
**commune à la voix** (médiane de toutes les syllabes du locuteur). Évaluez sur les mots fongbe dont le ton est écrit
(◌́ = haut, ◌̀ = bas). Quelle précision ? Quelles limites (contours, coarticulation, déclinaison) ?

In [ ]:
...  # ✍️ À COMPLÉTER

## ✅ Bilan
- La **F0** rend les tons visibles ; il faut la **normaliser** (demi-tons) pour comparer des voix.
- La **durée** distingue des mots en wolof ; la mesurer proprement demande de bien délimiter les sons.
- Les **formants** placent les voyelles dans un espace : c'est ce que « voit » un modèle dans le spectrogramme.
- À l'écrit, **Unicode** cache des pièges (NFC/NFD, homoglyphes ε/ɛ) : on **normalise** toujours avant de comparer.
- Une mesure objective permet aussi d'**évaluer** une synthèse vocale.

**Prochaine étape :** TP1 « Ma voix, ma langue » — enregistrez de vraies paires minimales et refaites ces analyses.